# SHIPP Stage 5A — Silver Listings Development

**Method:** Spark Window Current-State Reconstruction

```text
Bronze listing history
    ↓
partition by listing_id
    ↓
order by _pg_lsn DESC, _sort_by DESC
    ↓
select rn = 1
    ↓
retain current-row change types
    ↓
quality checks
    ↓
silver_listings
```

This notebook is for development and validation before promoting logic to
`data_pipeline/silver/silver_listings.py`.



```text
Bronze CDC history (pinned Delta version)
  → keep insert + update_postimage + delete
  → newest event per listing_id   (_pg_lsn DESC, _sort_by DESC)
  → drop keys whose newest event is delete
  → Silver contract + quality gate
  → overwrite Silver
  → reconcile Silver vs Bronze (two independent methods)
  → idempotency check
```

## How to run
1. Close any AI assistant panel that has auto-approve on.
2. **Run → Clear state and outputs**, then **Run all**. Never run single cells out of order.
3. Every gate either prints `PASS` or stops with an `AssertionError`. A failure means the data is wrong — fix the cause, never loosen the assert.

**Expected Silver content:** `demo-listing-001`, `demo-listing-002` — **2 rows**. `delete-test-listing-001` must NOT appear.

This notebook is the **only** writer of `silver_listings`.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# config

In [ ]:
CATALOG = "bootcamp_students"
CATALOG = "bootcamp_students"
SOURCE_TABLE = f"{CATALOG}.shipp_bronze.lb_listings_history"
TARGET_SCHEMA = f"{CATALOG}.shipp_silver"
TARGET_TABLE = f"{TARGET_SCHEMA}.silver_listings"
BUSINESS_KEY = "listing_id"

VALID_STATUSES = ["DRAFT", "AVAILABLE", "UNAVAILABLE", "WITHDRAWN", "EXPIRED"]

REQUIRED_SOURCE_COLUMNS = {
    "listing_id",
    "donor_id",
    "title",
    "description",
    "category",
    "condition",
    "latitude",
    "longitude",
    "available_from",
    "available_until",
    "status",
    "updated_at",
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

EXPECTED_SILVER_COLUMNS = [
    "listing_id",
    "donor_id",
    "title",
    "description",
    "category",
    "condition",
    "latitude",
    "longitude",
    "available_from",
    "available_until",
    "status",
    "updated_at",
    "source_pg_lsn",
    "source_sort_by",
    "source_change_type",
    "silver_processed_at",
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

print("SOURCE_TABLE =", SOURCE_TABLE)
print("TARGET_TABLE =", TARGET_TABLE)
print("BUSINESS_KEY =", BUSINESS_KEY)

# BRONZE_SCHEMA = "shipp_bronze"
# SILVER_SCHEMA = "shipp_silver"

# SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.lb_listings_history"
# TARGET_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA}"
# TARGET_TABLE = f"{TARGET_SCHEMA}.silver_listings"

# BUSINESS_KEY = "listing_id"

# VALID_CHANGE_TYPES = {
#     "insert",
#     "update_postimage",
# }

# VALID_LISTING_STATUSES = {
#     "DRAFT",
#     "AVAILABLE",
#     "UNAVAILABLE",
#     "WITHDRAWN",
#     "EXPIRED",
# }

# print("SOURCE_TABLE =", SOURCE_TABLE)
# print("TARGET_TABLE =", TARGET_TABLE)
# print("BUSINESS_KEY =", BUSINESS_KEY)

## Step 2 — Load Bronze (pinned) + source schema contract

In [ ]:
bronze_df, bronze_version = read_bronze_pinned(SOURCE_TABLE)

print("Bronze row count:", bronze_df.count())

missing_columns = REQUIRED_SOURCE_COLUMNS - set(bronze_df.columns)
assert not missing_columns, f"Bronze schema contract failed. Missing: {sorted(missing_columns)}"
print("PASS — Bronze source schema contract.")

display(bronze_df.groupBy("_pg_change_type").count().orderBy("_pg_change_type"))

 Delete-aware current-state reconstruction

In [ ]:
latest_events_df, current_rows_df, deleted_keys_df = reconstruct_current_state(bronze_df, BUSINESS_KEY)

print("Keys with any event:        ", latest_events_df.count())
print("Newest event = delete (out):", deleted_keys_df.count())
print("Current rows for Silver:    ", current_rows_df.count())

display(
    latest_events_df
    .select(BUSINESS_KEY, "_pg_change_type", "_pg_lsn", "_sort_by", "status")
    .orderBy(BUSINESS_KEY)
)

silver contract

In [ ]:
silver_df = current_rows_df.select(
    # Business columns
    F.col("listing_id"),
    F.col("donor_id"),
    F.col("title"),
    F.col("description"),
    F.col("category"),
    F.col("condition"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("available_from"),
    F.col("available_until"),
    F.col("status"),
    F.col("updated_at"),

    # Traceability columns
    F.col("_pg_lsn").alias("source_pg_lsn"),
    F.col("_sort_by").alias("source_sort_by"),
    F.col("_pg_change_type").alias("source_change_type"),

    # Silver processing metadata
    F.current_timestamp().alias("silver_processed_at"),
)

print("Silver candidate rows:", silver_df.count())
display(silver_df.orderBy(BUSINESS_KEY))

pre write quality gate

In [ ]:
row_count = silver_df.count()

print("Distinct statuses:", sorted(r["status"] for r in silver_df.select("status").distinct().collect()))

violations = {
    "duplicate listing_id": silver_df.groupBy(BUSINESS_KEY).count().filter(F.col("count") > 1).count(),
    "null listing_id": silver_df.filter(F.col("listing_id").isNull()).count(),
    "null donor_id": silver_df.filter(F.col("donor_id").isNull()).count(),
    "null category": silver_df.filter(F.col("category").isNull()).count(),
    "null status": silver_df.filter(F.col("status").isNull()).count(),
    "latitude out of range": silver_df.filter(
        F.col("latitude").isNotNull() & ~F.col("latitude").between(-90, 90)
    ).count(),
    "longitude out of range": silver_df.filter(
        F.col("longitude").isNotNull() & ~F.col("longitude").between(-180, 180)
    ).count(),
    "invalid status": silver_df.filter(~F.col("status").isin(VALID_STATUSES)).count(),
    "available_until before available_from": silver_df.filter(
        F.col("available_from").isNotNull()
        & F.col("available_until").isNotNull()
        & (F.col("available_until") < F.col("available_from"))
    ).count(),
    "delete event in Silver": silver_df.filter(
        ~F.col("source_change_type").isin(SILVER_SOURCE_CHANGE_TYPES)
    ).count(),
}

print("Silver row count:", row_count)
for name, count in violations.items():
    print(f"  {name:<40} {count}")

failed = {name: count for name, count in violations.items() if count}
assert row_count > 0, "QUALITY FAILED: Silver would contain zero rows."
assert not failed, f"QUALITY FAILED: {failed}"
print("PASS — pre-write quality gate.")

write silver +exact schema check

In [ ]:
written_row_count = write_silver(silver_df, TARGET_TABLE)
assert written_row_count == row_count, (
    f"WRITE VALIDATION FAILED: wrote {written_row_count}, validated {row_count}"
)
assert_exact_schema(TARGET_TABLE, EXPECTED_SILVER_COLUMNS)

Reconcile Persisted Silver Against Bronze

In [ ]:
assert_silver_matches_bronze(
    TARGET_TABLE, bronze_df, current_rows_df, deleted_keys_df, BUSINESS_KEY
)

demo_count = spark.table(TARGET_TABLE).filter(F.col(BUSINESS_KEY) == "demo-listing-001").count()
print("demo-listing-001 rows in Silver:", demo_count)
assert demo_count == 1, "FAILED: demo record must appear exactly once in Silver."
print("PASS — demo record present exactly once.")

## Step 8 — Idempotency (rewrite the same state, compare logically)

In [ ]:
run1 = logical_snapshot(TARGET_TABLE, BUSINESS_KEY)
write_silver(silver_df, TARGET_TABLE)
run2 = logical_snapshot(TARGET_TABLE, BUSINESS_KEY)

print("Run 1 rows:", len(run1), "| Run 2 rows:", len(run2))
assert run1 == run2, "IDEMPOTENCY FAILED: logical Silver contents changed on rewrite."
print("PASS — Silver rewrite is logically idempotent.")

Step 9 — Exit summary

Returns a JSON summary to a caller (the delete test runs this notebook with dbutils.notebook.run). When you run the notebook yourself you will see Notebook exited — that is the expected end.

In [ ]:
summary = {
    "table": TARGET_TABLE,
    "rows": spark.table(TARGET_TABLE).count(),
    "bronze_version": bronze_version,
    "status": "PASS",
}
print(summary)
dbutils.notebook.exit(json.dumps(summary))